# Amazon Market Insights\n\nA reproducible exploration of Amazon product listings, pricing, discounts, ratings and review volume. This dataset is a catalog/review snapshot, not transaction-level sales data.

In [ ]:
from pathlib import Path\nimport pandas as pd\nimport matplotlib.pyplot as plt\n\nfrom src.data_pipeline import load_dataset, clean_dataset, build_category_summary\n\nDATA_PATH = Path('data/raw/amazondataset.xlsx')\nOUTPUT_DIR = Path('data/processed')\nOUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
raw = load_dataset(DATA_PATH)\nprint('Raw shape:', raw.shape)\ndisplay(raw.head())\ndisplay(raw.isna().sum().sort_values(ascending=False).to_frame('missing_values'))

In [ ]:
clean = clean_dataset(raw)\nprint('Clean shape:', clean.shape)\nprint('Unique products:', clean['product_id'].nunique())\nprint('Categories:', clean['category'].nunique())\ndisplay(clean.describe(include='all').T)

In [ ]:
category_summary = build_category_summary(clean)\ndisplay(category_summary.head(15))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))\nclean['discount_percentage'].plot(kind='hist', bins=25, ax=axes[0], title='Discount distribution')\nclean['rating'].plot(kind='hist', bins=20, ax=axes[1], title='Product rating distribution')\naxes[0].set_xlabel('Discount (%)'); axes[1].set_xlabel('Rating (out of 5)')\nplt.tight_layout()

In [ ]:
top_categories = category_summary.head(10).sort_values('product_count')\nax = top_categories.plot.barh(x='category', y='product_count', figsize=(9, 6), legend=False, title='Top categories by unique product count')\nax.set_xlabel('Unique products'); ax.set_ylabel('Category'); plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))\nax.scatter(clean['discount_percentage'], clean['rating'], alpha=.45)\nax.set(xlabel='Discount (%)', ylabel='Rating', title='Discount vs. product rating')\nplt.tight_layout()

In [ ]:
clean.to_csv(OUTPUT_DIR / 'amazon_products_cleaned.csv', index=False)\ncategory_summary.to_csv(OUTPUT_DIR / 'category_summary.csv', index=False)\nprint('Saved cleaned data and category summary to', OUTPUT_DIR)

## Interpretation guardrails\n\n- `rating_count` is review/rating volume metadata, not verified units sold.\n- There are no transaction dates or order amounts, so sales trends, revenue, and growth cannot be calculated from this dataset.\n- Associations between discount and rating are descriptive and do not establish causation.